# CSS surface-code decoding

Construct a small GKP surface code, select one CSS sector, inspect one decode, and estimate the sector logical-failure rate.

In [ ]:
import Pkg
repo_root = isfile(joinpath(pwd(), "Project.toml")) ? pwd() : normpath(joinpath(pwd(), ".."))
Pkg.activate(repo_root)

using LinearAlgebra
using LatticeDecoder
using Random

samples_per_point = parse(Int, get(ENV, "LATTICEDECODER_EXAMPLE_SAMPLES", "20"))

In [ ]:
d = 3
basis = :X
code_definition = GKP_Surface_Code(d, false)
M = code_definition.code
half = size(M, 1) ÷ 2
Mqq = M[1:half, 1:half]
Mpp = M[(half + 1):end, (half + 1):end]

H, G = basis === :X ? (Mqq, inv(Mpp)) : (Mpp, -inv(Mqq))
logical_check = inv(H)
problem = QuantumDecodingProblem(H, G, logical_check)

sigma = 0.12
rng = MersenneTwister(30)
error_vector = sample_error(rng, sigma, size(H, 2))
received = copy(error_vector)
decoder = LDLCDecoder(
    initialize_tanner_graph(H);
    schedule=:serial,
    algorithm=:lsd,
    sigma,
    max_iterations=size(H, 2),
)
soft_estimate = run_decoder!(decoder, received)
decision = hard_decision(soft_estimate, H)
residual = error_vector - (received - G * decision)

(logical_error=is_logical_error(logical_check, residual), decision=decision)

In [ ]:
sigmas = [0.08, 0.12, 0.16]
estimates = [
    estimate_logical_error_rate!(
        MersenneTwister(300),
        LDLCDecoder(
            initialize_tanner_graph(H);
            schedule=:serial,
            algorithm=:lsd,
            sigma,
            max_iterations=size(H, 2),
        ),
        problem;
        samples=samples_per_point,
    )
    for sigma in sigmas
]

[
    (
        sigma=sigma,
        failures=result.events,
        samples=result.samples,
        rate=result.rate,
        interval=(result.lower, result.upper),
    )
    for (sigma, result) in zip(sigmas, estimates)
]

Repeat the construction with `basis = :Z` to decode the other CSS sector. Increase the sample count only after the smoke calculation completes.